<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Bubble Plots**


Estimated time needed: **30** minutes


In this lab, you will focus on visualizing data.

The dataset will be directly loaded into pandas for analysis and visualization.

You will use various visualization techniques to explore the data and uncover key trends.


## Objectives


In this lab, you will perform the following:


-   Visualize the distribution of data.

-   Visualize the relationship between two data features.

-   Visualize composition of data.

-   Visualize comparison of data.


#### Setup: Working with the Database
**Install and import the needed libraries**


In [ ]:
#!pip install pandas 
#!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt

**Download and connect to the database file containing survey data.**


To start, download and load the dataset into a `pandas` DataFrame.



In [ ]:
# Step 1: Download the dataset
#!wget -O survey-data.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

# Load the data
df = pd.read_csv("survey-data.csv")

# Display the first few rows of the data to understand its structure
df.head()


### Task 1: Exploring Data Distributions Using Bubble Plots


#### 1. Bubble Plot for Age vs. Frequency of Participation


- Visualize the relationship between respondents’ age and their participation frequency (`SOPartFreq`) using a bubble plot.

- Use the size of the bubbles to represent their job satisfaction (`JobSat`).


In [ ]:
##Write your code here
sub = df[['Age', 'SOPartFreq', 'JobSat']].dropna()
agg = sub.groupby(['Age', 'SOPartFreq'])['JobSat'].agg(['mean', 'count']).reset_index()

age_order = ['Under 18 years old', '18-24 years old', '25-34 years old', '35-44 years old', 
             '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say']
freq_order = ['Multiple times per day', 'Daily or almost daily', 'A few times per week', 
              'A few times per month or weekly', 'Less than once per month or monthly', 
              'I have never participated in Q&A on Stack Overflow']

agg['Age'] = pd.Categorical(agg['Age'], categories=age_order, ordered=True)
agg['SOPartFreq'] = pd.Categorical(agg['SOPartFreq'], categories=freq_order, ordered=True)
agg = agg.sort_values(['Age', 'SOPartFreq'])
agg['age_code'] = agg['Age'].cat.codes
agg['freq_code'] = agg['SOPartFreq'].cat.codes

plt.figure(figsize=(14, 8))
scatter = plt.scatter(agg['age_code'], agg['freq_code'],
                      s=agg['mean'] * 50, c=agg['count'], cmap='Blues',
                      alpha=0.7, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Respondent Count')
plt.xticks(ticks=range(len(age_order)), labels=age_order, rotation=45)
plt.yticks(ticks=range(len(freq_order)), labels=freq_order)
plt.xlabel('Age Group')
plt.ylabel('Participation Frequency')
plt.title('Age vs Participation Frequency (Bubble Size = Mean JobSat)')
plt.tight_layout()
plt.show()

#### 2. Bubble Plot for Compensation vs. Job Satisfaction


-Visualize the relationship between yearly compensation (`ConvertedCompYearly`) and job satisfaction (`JobSat`).

- Use the size of the bubbles to represent respondents’ age.


In [ ]:
##Write your code here
sub = df[['ConvertedCompYearly', 'JobSat', 'Age']].dropna()

# Map Age to numeric midpoint for averaging
age_map = {
    'Under 18 years old': 16, '18-24 years old': 21, '25-34 years old': 30,
    '35-44 years old': 40, '45-54 years old': 50, '55-64 years old': 60,
    '65 years or older': 70, 'Prefer not to say': None
}
sub['AgeNum'] = sub['Age'].map(age_map)
sub = sub.dropna(subset=['AgeNum'])

# Bin ConvertedCompYearly into quantile bins
sub['CompBin'] = pd.qcut(sub['ConvertedCompYearly'], q=8, duplicates='drop')

agg = sub.groupby(['CompBin', 'JobSat'], observed=False).agg(
    mean_age=('AgeNum', 'mean'), count=('AgeNum', 'count')
).reset_index()

# Create label for x-axis
agg['CompLabel'] = agg['CompBin'].astype(str)

plt.figure(figsize=(14, 8))
scatter = plt.scatter(agg['CompLabel'], agg['JobSat'],
                      s=agg['mean_age'] * 5, c=agg['count'], cmap='viridis',
                      alpha=0.7, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Respondent Count')
plt.xlabel('Yearly Compensation Range')
plt.ylabel('Job Satisfaction (0-10)')
plt.title('Compensation vs Job Satisfaction (Bubble Size = Mean Age)')
plt.xticks(rotation=45, fontsize=8)
plt.tight_layout()
plt.show()

### Task 2: Analyzing Relationships Using Bubble Plots


#### 1. Bubble Plot of Technology Preferences by Age

- Visualize the popularity of programming languages respondents have worked with (`LanguageHaveWorkedWith`) across age groups.

- Use bubble size to represent the frequency of each language.



In [ ]:
##Write your code here
sub = df[['Age', 'LanguageHaveWorkedWith']].dropna()
lang_exp = sub.assign(Language=sub['LanguageHaveWorkedWith'].str.split(';')).explode('Language')
lang_agg = lang_exp.groupby(['Age', 'Language'], observed=False).size().reset_index(name='count')

age_order = ['Under 18 years old', '18-24 years old', '25-34 years old', '35-44 years old', 
             '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say']
lang_agg['Age'] = pd.Categorical(lang_agg['Age'], categories=age_order, ordered=True)
lang_agg = lang_agg.sort_values('Age')

plt.figure(figsize=(16, 10))
scatter = plt.scatter(lang_agg['Age'], lang_agg['Language'],
                      s=lang_agg['count'] * 0.3, c=lang_agg['count'], cmap='plasma',
                      alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter, label='Respondent Count')
plt.xlabel('Age Group')
plt.ylabel('Programming Language')
plt.title('Programming Languages Worked With Across Age Groups')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### 2. Bubble Plot for Preferred Databases vs. Job Satisfaction

- Explore the relationship between preferred databases (`DatabaseWantToWorkWith`) and job satisfaction.

- Use bubble size to indicate the number of respondents for each database.


In [ ]:
##Write your code here
sub = df[['DatabaseWantToWorkWith', 'JobSat']].dropna()
db_exp = sub.assign(Database=sub['DatabaseWantToWorkWith'].str.split(';')).explode('Database')
db_agg = db_exp.groupby(['Database', 'JobSat'], observed=False).size().reset_index(name='count')

plt.figure(figsize=(16, 10))
scatter = plt.scatter(db_agg['Database'], db_agg['JobSat'],
                      s=db_agg['count'] * 0.3, c=db_agg['count'], cmap='coolwarm',
                      alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter, label='Respondent Count')
plt.xlabel('Preferred Database')
plt.ylabel('Job Satisfaction')
plt.title('Preferred Databases vs Job Satisfaction')
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

### Task 3: Comparing Data Using Bubble Plots


#### 1. Bubble Plot for Compensation Across Developer Roles

- Visualize compensation (`ConvertedCompYearly`) across different developer roles (`DevType`).

- Use bubble size to represent job satisfaction.


In [ ]:
##Write your code here
sub = df[['DevType', 'ConvertedCompYearly', 'JobSat']].dropna()
dev_exp = sub.assign(DevType=sub['DevType'].str.split(';')).explode('DevType')
dev_agg = dev_exp.groupby('DevType', observed=False).agg(
    mean_comp=('ConvertedCompYearly', 'mean'),
    mean_jobsat=('JobSat', 'mean'),
    count=('JobSat', 'count')
).reset_index()

plt.figure(figsize=(14, 10))
scatter = plt.scatter(dev_agg['DevType'], dev_agg['mean_comp'],
                      s=dev_agg['mean_jobsat'] * 50, c=dev_agg['count'], cmap='viridis',
                      alpha=0.7, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Respondent Count')
plt.xlabel('Developer Role')
plt.ylabel('Mean Yearly Compensation (USD)')
plt.title('Compensation Across Developer Roles (Bubble Size = Mean JobSat)')
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

#### 2. Bubble Plot for Collaboration Tools by Age

- Visualize the relationship between the collaboration tools used (`NEWCollabToolsHaveWorkedWith`) and age groups.

- Use bubble size to represent the frequency of tool usage.


In [ ]:
##Write your code here
sub = df[['Age', 'NEWCollabToolsHaveWorkedWith']].dropna()
tool_exp = sub.assign(Tool=sub['NEWCollabToolsHaveWorkedWith'].str.split(';')).explode('Tool')
tool_agg = tool_exp.groupby(['Age', 'Tool'], observed=False).size().reset_index(name='count')

age_order = ['Under 18 years old', '18-24 years old', '25-34 years old', '35-44 years old',
             '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say']
tool_agg['Age'] = pd.Categorical(tool_agg['Age'], categories=age_order, ordered=True)
tool_agg = tool_agg.sort_values('Age')

plt.figure(figsize=(16, 10))
scatter = plt.scatter(tool_agg['Age'], tool_agg['Tool'],
                      s=tool_agg['count'] * 0.3, c=tool_agg['count'], cmap='plasma',
                      alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter, label='Respondent Count')
plt.xlabel('Age Group')
plt.ylabel('Collaboration Tool')
plt.title('Collaboration Tools Used Across Age Groups')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Task 4: Visualizing Technology Trends Using Bubble Plots


#### 1. Bubble Plot for Preferred Web Frameworks vs. Job Satisfaction

- Explore the relationship between preferred web frameworks (`WebframeWantToWorkWith`) and job satisfaction.

- Use bubble size to represent the number of respondents.



In [ ]:
##Write your code here
sub = df[['WebframeWantToWorkWith', 'JobSat']].dropna()
wf_exp = sub.assign(Webframe=sub['WebframeWantToWorkWith'].str.split(';')).explode('Webframe')
wf_agg = wf_exp.groupby(['Webframe', 'JobSat'], observed=False).size().reset_index(name='count')

plt.figure(figsize=(16, 10))
scatter = plt.scatter(wf_agg['Webframe'], wf_agg['JobSat'],
                      s=wf_agg['count'] * 0.3, c=wf_agg['count'], cmap='coolwarm',
                      alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter, label='Respondent Count')
plt.xlabel('Preferred Web Framework')
plt.ylabel('Job Satisfaction')
plt.title('Preferred Web Frameworks vs Job Satisfaction')
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

#### 2. Bubble Plot for Admired Technologies Across Countries

- Visualize the distribution of admired technologies (`LanguageAdmired`) across different countries (`Country`).

- Use bubble size to represent the frequency of admiration.



In [ ]:
##Write your code here
sub = df[['LanguageAdmired', 'Country']].dropna()
lang_exp = sub.assign(Language=sub['LanguageAdmired'].str.split(';')).explode('Language')
lang_agg = lang_exp.groupby(['Country', 'Language'], observed=False).size().reset_index(name='count')

# Keep top 15 countries and top 15 languages for readability
top_countries = lang_agg.groupby('Country')['count'].sum().nlargest(15).index
top_langs = lang_agg.groupby('Language')['count'].sum().nlargest(15).index
lang_filt = lang_agg[lang_agg['Country'].isin(top_countries) & lang_agg['Language'].isin(top_langs)]

plt.figure(figsize=(18, 12))
scatter = plt.scatter(lang_filt['Country'], lang_filt['Language'],
                      s=lang_filt['count'] * 0.8, c=lang_filt['count'], cmap='magma',
                      alpha=0.7, edgecolors='k', linewidth=0.3)
plt.colorbar(scatter, label='Admiration Count')
plt.xlabel('Country')
plt.ylabel('Admired Language')
plt.title('Admired Technologies Across Countries')
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

## Final Step: Review


After completing the lab, you will have extensively used bubble plots to gain insights into developer community preferences, demographics, compensation trends, and job satisfaction.


## Summary


After completing this lab, you will be able to:

- Create and interpret bubble plots to analyze relationships and compositions within datasets.

- Use bubble plots to explore developer preferences, compensation trends, and satisfaction levels.

- Apply bubble plots to visualize complex relationships involving multiple dimensions effectively.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-10-29|1.2|Madhusudhan Moole|Updated lab|
|2024-10-16|1.1|Madhusudhan Moole|Updated lab|
|2024-10-15|1.0|Raghul Ramesh|Created lab|
--!>


Copyright © IBM Corporation. All rights reserved.
